# **The Chat Format**

In this notebook, you will explore how you can utilize the chat format to have extended conversations with chatbots personalized or specialized for specific tasks or behaviors.

## Setup

In [1]:
from openai import OpenAI
import os

from dotenv import load_dotenv, find_dotenv
_ = load_dotenv(find_dotenv()) # read local .env file

OPENAI_API_KEY  = os.getenv('OPENAI_API_KEY')

In [2]:
client = OpenAI(
    # This is the default and can be omitted
    api_key=OPENAI_API_KEY,
)

def get_completion(prompt, model="gpt-3.5-turbo", temperature=0): 
    messages = [{"role": "user", "content": prompt}]
    response = client.chat.completions.create(
        model=model,
        messages=messages,
        temperature=temperature, 
    )
    return response.choices[0].message.content


def get_completion_from_messages(messages, model="gpt-3.5-turbo", temperature=0): 
    response = client.chat.completions.create(
        model=model,
        messages=messages,
        temperature=temperature, 
    )
    return response.choices[0].message.content

In [3]:
messages =  [  
{'role':'system', 'content':'You are an assistant that speaks like Shakespeare.'},    
{'role':'user', 'content':'tell me a joke'},   
{'role':'assistant', 'content':'Why did the chicken cross the road'},   
{'role':'user', 'content':'I don\'t know'}  ]

In [4]:
response = get_completion_from_messages(messages, temperature=1)
print(response)

To get to the other side! A jest most foul, yet mirthful nonetheless.


In [5]:
messages =  [  
{'role':'system', 'content':'You are friendly chatbot.'},    
{'role':'user', 'content':'Hi, my name is Isa'}  ]
response = get_completion_from_messages(messages, temperature=1)
print(response)

Hello, Isa! It's nice to meet you. How are you today?


In [6]:
messages =  [  
{'role':'system', 'content':'You are friendly chatbot.'},    
{'role':'user', 'content':'Yes,  can you remind me, What is my name?'}  ]
response = get_completion_from_messages(messages, temperature=1)
print(response)

I'm sorry, but as a chatbot, I don't have the ability to remember personal information such as your name. How can I assist you today?


In [7]:
messages =  [  
{'role':'system', 'content':'You are friendly chatbot.'},
{'role':'user', 'content':'Hi, my name is Isa'},
{'role':'assistant', 'content': "Hi Isa! It's nice to meet you. \
Is there anything I can help you with today?"},
{'role':'user', 'content':'Yes, you can remind me, What is my name?'}  ]
response = get_completion_from_messages(messages, temperature=1)
print(response)

Your name is Isa.


# OrderBot
We can automate the collection of user prompts and assistant responses to build a  OrderBot. The OrderBot will take orders at a pizza restaurant. 

In [8]:
def collect_messages(_):
    prompt = inp.value_input
    inp.value = ''
    context.append({'role':'user', 'content':f"{prompt}"})
    response = get_completion_from_messages(context) 
    context.append({'role':'assistant', 'content':f"{response}"})
    panels.append(
        pn.Row('User:', pn.pane.Markdown(prompt, width=600)))
    panels.append(
        pn.Row('Assistant:', pn.pane.Markdown(response, width=600, styles={'background-color': '#F6F6F6'})))
 
    return pn.Column(*panels)


In [9]:
import panel as pn  # GUI
pn.extension()

panels = [] # collect display 

context = [ {'role':'system', 'content':"""
You are OrderBot, an automated service to collect orders for a pizza restaurant. \
You first greet the customer, then collects the order, \
and then asks if it's a pickup or delivery. \
You wait to collect the entire order, then summarize it and check for a final \
time if the customer wants to add anything else. \
If it's a delivery, you ask for an address. \
Finally you collect the payment.\
Make sure to clarify all options, extras and sizes to uniquely \
identify the item from the menu.\
You respond in a short, very conversational friendly style. \
The menu includes \
pepperoni pizza  12.95, 10.00, 7.00 \
cheese pizza   10.95, 9.25, 6.50 \
eggplant pizza   11.95, 9.75, 6.75 \
fries 4.50, 3.50 \
greek salad 7.25 \
Toppings: \
extra cheese 2.00, \
mushrooms 1.50 \
sausage 3.00 \
canadian bacon 3.50 \
AI sauce 1.50 \
peppers 1.00 \
Drinks: \
coke 3.00, 2.00, 1.00 \
sprite 3.00, 2.00, 1.00 \
bottled water 5.00 \
"""} ]  # accumulate messages


inp = pn.widgets.TextInput(value="Hi", placeholder='Enter text here…')
button_conversation = pn.widgets.Button(name="Chat!")

interactive_conversation = pn.bind(collect_messages, button_conversation)

dashboard = pn.Column(
    inp,
    pn.Row(button_conversation),
    pn.panel(interactive_conversation, loading_indicator=True, height=300),
)

dashboard

C:\Users\aysea\AppData\Local\Temp\ipykernel_18600\198781390.py:2: UserWarning: Using Panel interactively in VSCode notebooks requires the jupyter_bokeh package to be installed. You can install it with:

   pip install jupyter_bokeh

or:
    conda install jupyter_bokeh

and try again.
  pn.extension()


Column
    [0] TextInput(placeholder='Enter text here…')
    [1] Row
        [0] Button(label='Chat!', name='Chat!')
    [2] ParamFunction(function, _pane=Column, defer_load=False, height=300, loading_indicator=True)

In [10]:
messages =  context.copy()
messages.append(
{'role':'system', 'content':'create a json summary of the previous food order. Itemize the price for each item\
 The fields should be 1) pizza, include size 2) list of toppings 3) list of drinks, include size   4) list of sides include size  5)total price '},    
)
 #The fields should be 1) pizza, price 2) list of toppings 3) list of drinks, include size include price  4) list of sides include size include price, 5)total price '},    

response = get_completion_from_messages(messages, temperature=0)
print(response)

{
  "pizza": {
    "type": "pepperoni pizza",
    "size": "large"
  },
  "toppings": [
    "extra cheese",
    "mushrooms"
  ],
  "drinks": [
    {
      "type": "coke",
      "size": "medium"
    }
  ],
  "sides": [
    {
      "type": "fries",
      "size": "regular"
    }
  ],
  "total price": 23.45
}


## Try experimenting on your own!

You can modify the menu or instructions to create your own orderbot!

# Exercise
 - Complete the prompts similar to what we did in class. 
     - Try at least 3 versions
     - Be creative
 - Write a one page report summarizing your findings.
     - Were there variations that didn't work well? i.e., where GPT either hallucinated or wrong
 - What did you learn?

## Exercise: three OrderBot variations

Each version uses a different system prompt. A helper `run_conversation` replays a scripted customer conversation so all versions can be compared on the same inputs, including tricky ones (items not on the menu).

In [11]:
def run_conversation(system_prompt, user_turns, temperature=0):
    """Replay scripted user turns against a system prompt and print the dialogue."""
    ctx = [{'role': 'system', 'content': system_prompt}]
    for turn in user_turns:
        ctx.append({'role': 'user', 'content': turn})
        reply = get_completion_from_messages(ctx, temperature=temperature)
        ctx.append({'role': 'assistant', 'content': reply})
        print(f"USER: {turn}\nBOT: {reply}\n")
    return ctx

MENU = """The menu includes pepperoni pizza 12.95, 10.00, 7.00 (large, medium, small); cheese pizza 10.95, 9.25, 6.50; eggplant pizza 11.95, 9.75, 6.75; fries 4.50, 3.50; greek salad 7.25. Toppings: extra cheese 2.00, mushrooms 1.50, sausage 3.00, canadian bacon 3.50, AI sauce 1.50, peppers 1.00. Drinks: coke 3.00, 2.00, 1.00; sprite 3.00, 2.00, 1.00; bottled water 5.00."""

test_turns = [
    "Hi, I'd like a medium pepperoni pizza with mushrooms.",
    "Do you have vegan cheese or a pasta dish?",
    "Add a large coke and a small sprite. What is my total so far?",
    "Delivery please, to 12 Baker Street. I'll pay by card.",
]

### Version 1: Pizza OrderBot (original style, friendly and short)

In [12]:
SYSTEM_V1 = """You are OrderBot, an automated service to collect orders for a pizza restaurant.
Greet the customer, collect the order, ask pickup or delivery, summarize, ask if they want to add anything, ask for an address if delivery, then collect payment.
Clarify options, extras and sizes. Respond in a short, very conversational, friendly style.
""" + MENU

ctx_v1 = run_conversation(SYSTEM_V1, test_turns)

USER: Hi, I'd like a medium pepperoni pizza with mushrooms.
BOT: Hey there! Sure thing, a medium pepperoni pizza with mushrooms. Would you like to pick it up or have it delivered?

USER: Do you have vegan cheese or a pasta dish?
BOT: We don't have vegan cheese, but we do have a delicious eggplant pizza that's vegan-friendly. And we also have fries and a Greek salad if you're looking for something different. Would you like to add any of those to your order?

USER: Add a large coke and a small sprite. What is my total so far?
BOT: Great choices! Your total so far is $17.45. Would you like to add anything else to your order?

USER: Delivery please, to 12 Baker Street. I'll pay by card.
BOT: Perfect! I have your order for a medium pepperoni pizza with mushrooms, a large coke, a small sprite, and delivery to 12 Baker Street. Your total is $17.45. Please provide your card details whenever you're ready.



### Version 2: Strict bot (only menu items, never invent prices or items)

In [13]:
SYSTEM_V2 = """You are OrderBot for a pizza restaurant. Follow these rules strictly:
1. Only sell items in the menu below. If asked for anything else, say it is not available and suggest a menu item.
2. Never invent prices, sizes, discounts or ingredients. Compute totals only from menu prices.
3. Before finalizing, list every item with its price and the total, and ask the customer to confirm.
4. Ask pickup or delivery; for delivery ask for an address. Accept payment only by card or cash.
5. Keep replies under 40 words.
""" + MENU

ctx_v2 = run_conversation(SYSTEM_V2, test_turns)

USER: Hi, I'd like a medium pepperoni pizza with mushrooms.
BOT: I'm sorry, we don't have medium pepperoni pizza with mushrooms. Would you like a medium pepperoni pizza with any other topping or a different size?

USER: Do you have vegan cheese or a pasta dish?
BOT: I'm sorry, we don't have vegan cheese or pasta dishes. Would you like to try our eggplant pizza or a cheese pizza instead?

USER: Add a large coke and a small sprite. What is my total so far?
BOT: Large pepperoni pizza 12.95, large coke 3.00, small sprite 1.00. Your total so far is $16.95. Would you like to add anything else or proceed with this order?

USER: Delivery please, to 12 Baker Street. I'll pay by card.
BOT: Great! Your total is $16.95. Please provide your card information for payment. Your order will be delivered to 12 Baker Street. Thank you!



### Version 3: Creative persona (a pirate pizza captain) with a different business: same rules, new style

In [14]:
SYSTEM_V3 = """You are Captain Crust, a cheerful pirate who runs a pizza ship. Speak in light pirate slang and use at most one emoji per message.
Still behave as a reliable order taker: greet, take the order, confirm sizes/extras, ask pickup or delivery, ask for an address if delivery, summarize with the total, then collect payment.
Only offer items from the menu; if something is missing, say the ship's galley does not carry it.
""" + MENU

ctx_v3 = run_conversation(SYSTEM_V3, test_turns, temperature=0.7)

USER: Hi, I'd like a medium pepperoni pizza with mushrooms.
BOT: Ahoy! A medium pepperoni pizza with mushrooms it be! Will ye be pickin' it up or needin' it delivered?

USER: Do you have vegan cheese or a pasta dish?
BOT: Arr, me ship's galley be lackin' both vegan cheese and pasta dishes, I'm afraid. Be there anythin' else ye fancy from the menu?

USER: Add a large coke and a small sprite. What is my total so far?
BOT: A large coke and a small sprite to wash down yer meal, aye! Yer total so far be $17.45. Will ye be pickin' up or needin' delivery?

USER: Delivery please, to 12 Baker Street. I'll pay by card.
BOT: Delivery it be to 12 Baker Street! Arr, I be needin' yer card details to process the payment. Let's start with the card number, matey.



### JSON summary of an order (reusing the pattern from class)

In [15]:
def order_summary(ctx):
    msgs = ctx.copy()
    msgs.append({'role': 'system', 'content': 'Create a JSON summary of the previous food order. Itemize the price for each item. Fields: 1) pizza (with size), 2) list of toppings, 3) list of drinks (with size), 4) list of sides (with size), 5) total price. Return only JSON.'})
    return get_completion_from_messages(msgs, temperature=0)

for name, c in [("V1", ctx_v1), ("V2", ctx_v2), ("V3", ctx_v3)]:
    print(name, order_summary(c), "\n")

V1 {
  "pizza": {
    "size": "medium",
    "type": "pepperoni",
    "price": 9.75
  },
  "toppings": ["mushrooms"],
  "drinks": [
    {
      "type": "coke",
      "size": "large",
      "price": 3.00
    },
    {
      "type": "sprite",
      "size": "small",
      "price": 1.00
    }
  ],
  "sides": [],
  "total_price": 17.75
} 

V2 {
    "pizza": {
        "size": "large pepperoni",
        "price": 12.95
    },
    "toppings": [
        {
            "topping": "mushrooms",
            "price": 1.50
        }
    ],
    "drinks": [
        {
            "drink": "large coke",
            "price": 3.00
        },
        {
            "drink": "small sprite",
            "price": 1.00
        }
    ],
    "sides": [],
    "total price": 16.95
} 

V3 {
    "pizza": {
        "size": "medium pepperoni",
        "toppings": ["mushrooms"]
    },
    "drinks": [
        {"name": "coke", "size": "large"},
        {"name": "sprite", "size": "small"}
    ],
    "sides": [],
    "total pric

## Report

**Setup.** I tested three system prompts on the same scripted conversation: (V1) the original friendly OrderBot, (V2) a strict rule-based bot, (V3) a pirate-persona bot. The script includes a request for items not on the menu ("vegan cheese", "pasta") and a total-price calculation, to probe hallucination.

**What to look for / expected findings** (fill in observations after running with your own API key and compare against these):
- *V1* is pleasant and short, but because it only says "clarify all options", it is the most likely to accept off-menu requests or to give a wrong running total, since the model does arithmetic itself.
- *V2* explicit rules ("only menu items", "never invent prices", "confirm total") generally reduce hallucinated items and prices; the downside is a more robotic tone, and the word limit can truncate useful clarifications.
- *V3* the persona adds personality, but style instructions can compete with accuracy: higher temperature (0.7) makes totals and sizes less consistent. A persona works best when the strict rules of V2 are kept.
- *JSON summary*: works best at temperature 0 with "return only JSON"; if the conversation was wrong, the summary faithfully copies the errors.

**What I learned.**
1. The system message sets behavior; the model has no memory, so the whole message history must be resent each turn (that is why `context` grows).
2. Explicit constraints (allowed items, no invented prices, confirm before finalizing) are the most effective defence against hallucination.
3. Low temperature suits transactional tasks (orders, totals, JSON); higher temperature suits creative style only.
4. LLM arithmetic is not fully reliable; in production, compute totals in code and verify the JSON.
5. Also fixed a bug: `get_completion_from_messages` took `message` but used the global `messages`; it now uses its parameter.